In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [23]:
original_df = pd.read_csv('./dummy_silver_data.csv')
original_df.head()

,location_id,sensor_id,location_name,datetime_utc,lat,lon,parameter,value,unit
0,AU001,S0001,Sydney - Rozelle,2026-08-06T00:00:00Z,-33.8608,151.1719,o3,8.59,µg/m³
1,AU001,S0001,Sydney - Rozelle,2026-08-06T00:00:00Z,-33.8608,151.1719,no2,17.90,µg/m³
2,AU001,S0001,Sydney - Rozelle,2026-08-06T00:00:00Z,-33.8608,151.1719,pm25,9.06,µg/m³
3,AU001,S0001,Sydney - Rozelle,2026-08-06T01:00:00Z,-33.8608,151.1719,o3,17.31,µg/m³
4,AU001,S0001,Sydney - Rozelle,2026-08-06T01:00:00Z,-33.8608,151.1719,no2,20.44,µg/m³


In [24]:
df = original_df.copy()

In [25]:
df["datetime_utc"] = pd.to_datetime(df["datetime_utc"], errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")

In [5]:
df = df.sort_values(
        ["sensor_id", "parameter", "datetime_utc"]
    )

In [6]:
# Calculate the station's rolling mean (last 7 observations, at least 3 required) for each parameter
df["rolling_mean_7"] = (
    df.groupby(["sensor_id", "parameter"])["value"]
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=3
        ).mean()
    )
)

In [7]:
df[[
    "sensor_id",
    "datetime_utc",
    "parameter",
    "value",
    "rolling_mean_7"
]].head(20)

,sensor_id,datetime_utc,parameter,value,rolling_mean_7
1,S0001,2026-08-06T00:00:00Z,no2,17.90,NaN
4,S0001,2026-08-06T01:00:00Z,no2,20.44,NaN
7,S0001,2026-08-06T02:00:00Z,no2,31.51,23.283333
10,S0001,2026-08-06T03:00:00Z,no2,0.00,17.462500
13,S0001,2026-08-06T04:00:00Z,no2,11.67,16.304000
16,S0001,2026-08-06T05:00:00Z,no2,23.85,17.561667
19,S0001,2026-08-06T06:00:00Z,no2,4.83,15.742857
22,S0001,2026-08-06T07:00:00Z,no2,35.20,18.214286
25,S0001,2026-08-06T08:00:00Z,no2,26.53,19.084286
28,S0001,2026-08-06T09:00:00Z,no2,32.78,19.265714


In [26]:
dt = df["datetime_utc"]
df["hour"] = dt.dt.hour
df["dow"] = dt.dt.dayofweek
df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
df["month"] = dt.dt.month
# season keyed to Sydney (Southern Hemisphere)
df["season"] = df["month"] % 12 // 3 + 1  # 1=summer(DJF)...4=spring(SON) approx
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

In [27]:
group_keys = ["location_id", "sensor_id", "parameter", "hour", "is_weekend", "season"]
baseline = (
    df.groupby(group_keys)["value"]
    .agg(baseline_mean="mean", baseline_std="std", baseline_n="count")
    .reset_index()
)

In [28]:
baseline.shape

(2304, 9)

In [29]:
baseline.head()

,location_id,sensor_id,parameter,hour,is_weekend,season,baseline_mean,baseline_std,baseline_n
0,AU001,S0001,no2,0,0,3,14.259091,10.118315,11
1,AU001,S0001,no2,0,1,3,8.637500,7.146996,4
2,AU001,S0001,no2,1,0,3,18.756000,10.422386,10
3,AU001,S0001,no2,1,1,3,12.535000,6.070906,4
4,AU001,S0001,no2,2,0,3,12.807000,8.478801,10
